# YoNoSplat on Colab — pose-free feed-forward 3DGS (A/B vs AnySplat)

Test whether **YoNoSplat** (ICLR'26, cvg/YoNoSplat) beats our AnySplat feed-forward in FREE-ORBIT realism.
Pose-free, MIT code, **Pi3 NOT needed at inference** (only training) -> commercially clean to run.

**Two known risks:** (1) only **224x224** checkpoints are released (vs our AnySplat 448) -> may be blurrier;
(2) the `diff-gaussian-rasterization-w-pose` CUDA rasterizer must compile. We judge the result on the
free-orbit screenshot dome, not PSNR. Runtime -> **GPU (A100/L4)**, run cells in order.

## 0. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install conda (restarts the kernel ~2 min) — then continue from cell 2

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # kernel auto-restarts

## 2. Clone + build the YoNoSplat env (~10-15 min)
Python 3.10, torch 2.1.2 **cu121** (matches Colab's CUDA 12 so the rasterizer compiles; README's cu118
also works but mismatches system nvcc). Pi3 weights are skipped (inference doesn't need them).

In [ ]:
import condacolab; condacolab.check()
import os, subprocess
os.chdir('/content')
if not os.path.isdir('/content/YoNoSplat'):
    !git clone -q https://github.com/cvg/YoNoSplat.git
%cd /content/YoNoSplat
!conda create -n yono python=3.10 -y -q
# torch 2.1.2 cu121 (cu121 build matches Colab system CUDA -> rasterizer compiles)
!conda run -n yono pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
# requirements incl. the git rasterizer; build it against the system CUDA toolkit
!conda run -n yono bash -c 'export CUDA_HOME=/usr/local/cuda PATH=$CUDA_HOME/bin:$PATH TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6;8.9"; pip install -q -r requirements.txt'
print('\n--- verifying imports ---')
chk = subprocess.run(['conda','run','-n','yono','python','-c',
  'import torch,diff_gaussian_rasterization,lightning,hydra;print("torch",torch.__version__,"| rasterizer OK | cuda",torch.cuda.is_available())'],
  capture_output=True, text=True)
print(chk.stdout or chk.stderr)
# re10k checkpoint (indoor real-estate -> best match for a room) — Pi3 NOT downloaded (inference doesn't use it)
!mkdir -p pretrained_weights
!wget -q -O pretrained_weights/re10k_224x224_ctx2to32.ckpt https://huggingface.co/botaoye/YoNoSplat/resolve/main/re10k_224x224_ctx2to32.ckpt
!ls -la pretrained_weights/

## 3. Upload the packed scene chunk
Upload `datasets/hires_chunk.zip` (built locally by `tools/frames_to_re10k.py` — 48 frames @ 384px
square in re10k chunk format). It unpacks to `datasets/hires/`.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                      # pick datasets/hires_chunk.zip
zname = next(iter(up))
os.makedirs('/content/YoNoSplat/datasets', exist_ok=True)
with zipfile.ZipFile(zname) as z: z.extractall('/content/YoNoSplat/datasets')
print('extracted ->', os.listdir('/content/YoNoSplat/datasets/hires'))
assert os.path.isfile('/content/YoNoSplat/datasets/hires/test/000000.torch')
assert os.path.isfile('/content/YoNoSplat/datasets/hires/index_eval.json')

## 4. Run pose-free inference -> 3DGS PLY
`pose_free=true` + predicted intrinsics (the stored poses are dummies, ignored). 32 context views.
`save_debug_info=true` writes the standard INRIA SH0 PLY. ~1-2 min on A100.

In [ ]:
%cd /content/YoNoSplat
!conda run -n yono --no-capture-output python -m src.main \
  +experiment=yono_re10k mode=test wandb.mode=disabled wandb.name=yono_hires \
  dataset/view_sampler@dataset.re10k.view_sampler=evaluation \
  dataset.re10k.view_sampler.index_path=datasets/hires/index_eval.json \
  dataset.re10k.roots=[datasets/hires] \
  dataset.re10k.original_image_shape=[384,384] \
  dataset.re10k.view_sampler.num_context_views=32 \
  checkpointing.load=pretrained_weights/re10k_224x224_ctx2to32.ckpt \
  model.encoder.pose_free=true \
  model.encoder.backbone.use_pred_intrinsics_for_embed=true \
  test.align_pose=true test.save_debug_info=true test.save_image=true \
  model.decoder.prune_opacity_threshold=0.005
import glob
plys = glob.glob('/content/YoNoSplat/outputs/**/*_gaussians.ply', recursive=True)
print('\nPLY files:', plys)
if plys:
    import shutil; shutil.copyfile(plys[0], '/content/yono_hires.ply')
    from google.colab import files; files.download('/content/yono_hires.ply')
else:
    print('NO PLY produced — check the log above (likely the chunk format or pose-free path).')